# T.pcg joint stackups — promoters and cCREs

Renders the two joint H3K27me3 + H3K27ac stackup figures and their accessory
panels from the caches written by `10_promoter_stackup.py` and
`11_ccre_stackup.py`. Nothing here re-reads bigwigs, so the bands and curves
match the clustering exactly.

Each **split figure** puts every k-means cluster's mean ±5 kb meta-profile
directly under that cluster's stackup band, column-aligned to the stages — one
stage-coloured curve per column, with y shared within a mark so curve height
tracks heatmap intensity. Low-H3K27me3 clusters zoom to `ME3_ZOOM` so their
background rise is visible.

The left annotation panel follows the cache: promoters get a per-cluster
HB_vs_DE expression-LFC raincloud (`lfc_ord`), cCREs get a cCRE-class strip
(`cls_ord`).

In [ ]:
from pathlib import Path
import json

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch, Rectangle
from scipy.stats import gaussian_kde

mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['svg.fonttype'] = 'none'

## Config

In [ ]:
# Resolve this analysis folder (paper/05_ipt) regardless of the kernel's cwd.
def _here():
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / '01_ipt_clustering.py').exists():
            return c
    raise RuntimeError('run this notebook from inside paper/05_ipt')

DIR = _here()
RESULTS = DIR / 'results'
FIGS = DIR / 'figs'
FIGS.mkdir(parents=True, exist_ok=True)

PROMOTER_TAG = 'promoter_joint_cluster'   # <- 10_promoter_stackup.py
CCRE_TAG = 'ccre_joint_cluster'           # <- 11_ccre_stackup.py

POSITION = 'bottom'   # meta-profile above ('top') or below ('bottom') each band
PROF_FRAC = 0.07      # profile-row height as a fraction of total element count
ME3_ZOOM = 2.5

# canonical palette (vendored in data/), re-keyed onto this module's stage names
_RENAME = {'iHEP': 'iHLC', 'mHEP': 'HLC'}
STAGE_COLOR = {_RENAME.get(k, k): v for k, v in
               json.loads((DIR / 'data' / 'stage_colors.json').read_text()).items()}
MARK_CMAP = {'H3K27me3': 'magma', 'H3K27ac': 'viridis'}
# official SCREEN cCRE colours (CA-TF/TF de-emphasised to grey)
CLASS_COLOR = {'PLS': '#FF0000', 'pELS': '#FFA700', 'dELS': '#FFCD00',
               'CA-H3K4me3': '#FFAAAA', 'CA-CTCF': '#00B0F0', 'CA': '#06DA93',
               'CA-TF': '#9E9E9E', 'TF': '#6B6B6B'}

for _tag in (PROMOTER_TAG, CCRE_TAG):
    for _kind in ('stackup', 'metaprofiles'):
        _p = RESULTS / f'{_tag}_{_kind}.npz'
        if not _p.exists():
            raise SystemExit(f'missing {_p} — run 10_promoter_stackup.py '
                             f'and 11_ccre_stackup.py first')

## Split-figure renderer

In [ ]:
def raincloud(ax, d, nb):
    """One band's HB_vs_DE LFC raincloud: trimmed half-KDE on top, raw points below,
    median dot + IQR bar on the band centre line. `d` is in heatmap row order; the
    band spans rows [0, nb] on a top-down y-axis."""
    c, unit = nb / 2, nb
    pad, vio, jit = 0.03 * unit, 0.42 * unit, 0.40 * unit
    d = np.asarray(d, float)
    dv = d[np.isfinite(d)]
    if dv.size == 0:
        return
    if dv.size >= 3 and np.ptp(dv) > 0:
        xs = np.linspace(dv.min(), dv.max(), 256)
        dy = gaussian_kde(dv)(xs); dy = dy / dy.max() * vio
        ax.fill_between(xs, c - pad - dy, c - pad, color='0.82', edgecolor='0.45', lw=0.7, zorder=2)
    yj = c + pad + (np.arange(d.size) / max(d.size - 1, 1)) * jit
    ax.scatter(dv, yj[np.isfinite(d)], s=4, color='0.30', alpha=0.35, linewidths=0, zorder=3)
    q1, med, q3 = np.percentile(dv, [25, 50, 75])
    ax.plot([q1, q3], [c, c], color='0.1', lw=1.4, solid_capstyle='butt', zorder=4)
    ax.plot([med], [c], 'o', ms=4.5, mfc='white', mec='0.1', mew=1.0, zorder=5)


def split_figure(tag, position=None, prof_frac=None, me3_zoom=None, save=True):
    """Per-cluster meta-profiles interleaved with the stackup bands."""
    position = position or POSITION
    prof_frac = PROF_FRAC if prof_frac is None else prof_frac
    me3_zoom = ME3_ZOOM if me3_zoom is None else me3_zoom

    st = np.load(RESULTS / f'{tag}_stackup.npz', allow_pickle=True)
    mp = np.load(RESULTS / f'{tag}_metaprofiles.npz', allow_pickle=True)
    heat, sizes, bounds = st['heat'], st['sizes'], st['bounds']
    marks = [str(m) for m in st['marks']]
    stages = [str(s) for s in st['stages']]
    prof, sem, x = mp['profiles'], mp['sems'], mp['x']
    M, S, n, NB = heat.shape
    K = len(sizes)
    edges = np.concatenate([[0], np.cumsum(sizes)]).astype(int)
    dx = float(x[1] - x[0])
    xlo, xhi = x[0] - dx / 2, x[-1] + dx / 2

    mode = 'class' if 'cls_ord' in st.files else ('lfc' if 'lfc_ord' in st.files else 'none')
    cls_ord = st['cls_ord'] if mode == 'class' else None
    lfc_ord = st['lfc_ord'] if mode == 'lfc' else None
    lfc_xlim = float(np.nanmax(np.abs(lfc_ord)) * 1.05) if mode == 'lfc' else None

    vmax = ([float(v) for v in st['vmax']] if 'vmax' in st.files else
            [10.0 if marks[mi] == 'H3K27me3' else float(np.nanpercentile(heat[mi], 99))
             for mi in range(M)])
    ymax = [float(np.nanmax(prof[:, mi] + sem[:, mi]) * 1.05) for mi in range(M)]
    me3_peak = np.array([float(np.nanmax(prof[b, 0] + sem[b, 0])) for b in range(K)])

    def prof_ymax(b, mi):
        return me3_zoom if (mi == 0 and me3_peak[b] <= me3_zoom) else ymax[mi]

    lead = ({'class': [('panel', 0.25), ('spacer', 0.25)],
             'lfc': [('panel', 2.4), ('spacer', 0.18)],
             'none': [('spacer', 0.25)]})[mode]
    width_ratios = [w for _, w in lead] + [1] * S + [0.4] + [1] * S
    PANEL = next((i for i, (k, _) in enumerate(lead) if k == 'panel'), None)
    nlead = len(lead)
    me3_cols = list(range(nlead, nlead + S))
    ac_cols = list(range(nlead + S + 1, nlead + S + 1 + S))
    stage_cols = me3_cols + ac_cols
    col_ms = {c: (0, si) for si, c in enumerate(me3_cols)}
    col_ms.update({c: (1, si) for si, c in enumerate(ac_cols)})
    ncols = len(width_ratios)

    total = float(sizes.sum())
    p, g = prof_frac * total, 0.018 * total
    row_ratios, rows = [], []
    for i in range(K):
        band = ([('prof', i), ('heat', i)] if position == 'top' else [('heat', i), ('prof', i)])
        for kind, b in band:
            rows.append((kind, b)); row_ratios.append(p if kind == 'prof' else float(sizes[b]))
        if i < K - 1:
            rows.append(('gap', i)); row_ratios.append(g)

    fw = 0.95 * (2 * S) + (5.5 if mode == 'lfc' else 4)
    fig = plt.figure(figsize=(fw, 10))
    gs = fig.add_gridspec(len(rows), ncols, width_ratios=width_ratios,
                          height_ratios=row_ratios, wspace=0.08, hspace=0.0)

    top_row_ax, me3_ax, ac_ax, last_im = {}, [], [], {}

    def band_label(ax, b):
        ax.set_ylabel(f'k{b}\nn={int(sizes[b])}', fontsize=8, rotation=0,
                      ha='right', va='center', labelpad=8)

    for r, (kind, b) in enumerate(rows):
        if kind == 'gap':
            continue
        e0, e1 = edges[b], edges[b + 1]
        nb = e1 - e0
        if kind == 'heat':
            if mode == 'class':
                ax = fig.add_subplot(gs[r, PANEL])
                i0 = 0
                while i0 < nb:
                    j = i0
                    while j < nb and cls_ord[e0 + j] == cls_ord[e0 + i0]:
                        j += 1
                    ax.add_patch(Rectangle((0, i0), 1, j - i0, antialiased=False,
                                 facecolor=CLASS_COLOR.get(str(cls_ord[e0 + i0]), '#888'),
                                 edgecolor='none'))
                    i0 = j
                ax.set_xlim(0, 1); ax.set_ylim(nb, 0); ax.set_xticks([]); ax.set_yticks([])
                band_label(ax, b); top_row_ax.setdefault(PANEL, ax)
            elif mode == 'lfc':
                ax = fig.add_subplot(gs[r, PANEL])
                raincloud(ax, lfc_ord[e0:e1], nb)
                ax.axvline(0, color='0.5', lw=0.8, zorder=1)
                ax.set_ylim(nb - 0.5, -0.5); ax.set_yticks([])
                ax.set_xlim(-lfc_xlim, lfc_xlim)
                ax.spines[['top', 'right', 'left']].set_visible(False)
                if r == 0:
                    ax.set_title('expression Δ', fontsize=8)
                if b == K - 1:
                    ax.set_xlabel('HB_vs_DE LFC', fontsize=8); ax.tick_params(labelsize=7)
                else:
                    ax.set_xticklabels([])
                band_label(ax, b); top_row_ax.setdefault(PANEL, ax)
            for c in stage_cols:
                mi, si = col_ms[c]
                ax = fig.add_subplot(gs[r, c])
                im = ax.imshow(heat[mi, si, e0:e1], aspect='auto', cmap=MARK_CMAP[marks[mi]],
                               vmin=0, vmax=vmax[mi], interpolation='nearest')
                ax.axvline(NB / 2 - 0.5, color='white', lw=0.4, ls='--', alpha=0.5)
                ax.set_xticks([]); ax.set_yticks([])
                top_row_ax.setdefault(c, ax)
                last_im[mi] = im
                (me3_ax if mi == 0 else ac_ax).append(ax)
                if mode == 'none' and c == me3_cols[0]:
                    band_label(ax, b)
        else:
            for c in stage_cols:
                mi, si = col_ms[c]
                ax = fig.add_subplot(gs[r, c])
                y, ee = prof[b, mi, si], sem[b, mi, si]
                col = STAGE_COLOR.get(stages[si], '#444')
                ax.plot(x, y, color=col, lw=1.4)
                ax.fill_between(x, y - ee, y + ee, color=col, alpha=0.18, lw=0)
                ax.axvline(0, color='0.6', lw=0.5, ls='--')
                ax.set_ylim(0, prof_ymax(b, mi)); ax.set_xlim(xlo, xhi)
                ax.set_xticks([])
                for sp in ('top', 'right'):
                    ax.spines[sp].set_visible(False)
                if c in (me3_cols[0], ac_cols[0]):
                    ax.tick_params(axis='y', labelsize=6, length=2)
                else:
                    ax.set_yticks([]); ax.spines['left'].set_visible(False)
                top_row_ax.setdefault(c, ax)
                (me3_ax if mi == 0 else ac_ax).append(ax)

    for c in stage_cols:
        _, si = col_ms[c]
        top_row_ax[c].set_title(stages[si], fontsize=8, color=STAGE_COLOR[stages[si]],
                                fontweight='bold', pad=4)
    if mode == 'class':
        top_row_ax[PANEL].set_title('class', fontsize=7, rotation=90, va='bottom')

    for cols, name in ((me3_cols, 'H3K27me3'), (ac_cols, 'H3K27ac')):
        xc = np.mean([gs[0, c].get_position(fig).x0 + gs[0, c].get_position(fig).x1
                      for c in cols]) / 2
        fig.text(xc, 0.945, name, ha='center', fontsize=11, fontweight='bold')

    # pass BOTH heat + profile axes per block so the colorbar shrinks them by the same
    # transform and the columns stay aligned (shrinking only the heatmaps offsets them)
    cb = fig.colorbar(last_im[0], ax=me3_ax, fraction=0.015, pad=0.02)
    cb.set_label('H3K27me3 fc', fontsize=8)
    cb = fig.colorbar(last_im[1], ax=ac_ax, fraction=0.015, pad=0.02)
    cb.set_label('H3K27ac fc', fontsize=8)

    if mode == 'class':
        present = [c for c in CLASS_COLOR if c in set(map(str, cls_ord))]
        fig.legend(handles=[Patch(color=CLASS_COLOR[c], label=c) for c in present],
                   loc='lower center', ncol=len(present), fontsize=7, frameon=False,
                   bbox_to_anchor=(0.5, -0.01))
    fig.suptitle(f'{tag} — per-cluster meta-profiles {position} of stackups (k={K})',
                 fontsize=12, fontweight='bold', y=0.98)

    if save:
        stem = FIGS / f'{tag}_split'
        fig.savefig(f'{stem}.pdf', bbox_inches='tight')
        fig.savefig(f'{stem}.png', dpi=170, bbox_inches='tight')
        print(f'-> {stem.name}  (K={K}, mode={mode}, position={position})')
    return fig

## Heatmap 1 — T.pcg gene promoters

Left panel is the per-cluster HB_vs_DE expression-LFC raincloud.

In [ ]:
fig = split_figure(PROMOTER_TAG)

## Heatmap 2 — T.pcg cCREs

Left panel is the cCRE-class strip.

In [ ]:
fig = split_figure(CCRE_TAG)

## Accessory — cCRE-class composition per cluster

In [ ]:
def composition(classes):
    """Ordered (class, count) for the classes present, in canonical SCREEN order."""
    vals, cnts = np.unique(classes, return_counts=True)
    d = dict(zip(map(str, vals), cnts))
    return [(c, int(d.get(c, 0))) for c in CLASS_COLOR if d.get(c, 0) > 0]


def draw_pie(ax, classes, title):
    comp = composition(classes)
    labels = [c for c, _ in comp]
    counts = np.array([n for _, n in comp], float)
    ax.pie(counts, colors=[CLASS_COLOR[c] for c in labels], startangle=90,
           counterclock=False,
           autopct=lambda pct: f'{pct:.0f}%' if pct >= 4 else '',   # declutter small wedges
           pctdistance=0.72, wedgeprops=dict(edgecolor='white', linewidth=0.8),
           textprops=dict(fontsize=7, color='black'))
    ax.set_title(f'{title}\nn={int(counts.sum())}', fontsize=10, fontweight='bold')
    ax.set_aspect('equal')


st = np.load(RESULTS / f'{CCRE_TAG}_stackup.npz', allow_pickle=True)
cls_ord = np.array([str(c) for c in st['cls_ord']])
sizes = st['sizes']
edges = np.concatenate([[0], np.cumsum(sizes)]).astype(int)
K = len(sizes)

fig, axes = plt.subplots(1, K + 1, figsize=(3.1 * (K + 1), 3.6))
for b in range(K):
    draw_pie(axes[b], cls_ord[edges[b]:edges[b + 1]], f'k{b}')
draw_pie(axes[K], cls_ord, 'All')

present = [c for c in CLASS_COLOR if c in set(cls_ord)]
fig.legend(handles=[Patch(facecolor=CLASS_COLOR[c], edgecolor='white', label=c)
                    for c in present],
           loc='lower center', ncol=len(present), fontsize=8, frameon=False,
           bbox_to_anchor=(0.5, -0.06))
fig.suptitle(f'{CCRE_TAG} — cCRE-class composition per cluster', fontsize=12,
             fontweight='bold', y=1.02)
fig.tight_layout()
fig.savefig(FIGS / f'{CCRE_TAG}_class_pies.pdf', bbox_inches='tight')
fig.savefig(FIGS / f'{CCRE_TAG}_class_pies.png', dpi=170, bbox_inches='tight')

for b in range(K):
    comp = composition(cls_ord[edges[b]:edges[b + 1]])
    tot = sum(n for _, n in comp)
    print(f'  k{b} (n={tot}): ' + '  '.join(f'{c} {100*n/tot:.0f}%' for c, n in comp))
fig